We'll use cross-attention instead of self-attention to be more explicit about
which axis is softmax applied on.

We'll use the convention of row-major matrix in consistency with the
[paper](https://arxiv.org/abs/1706.03762) (Equation 1) and matrix representation
in torch/numpy.

Denote

$$
\begin{align}
\underset{n_q \times d_k}{Q}
&= \begin{bmatrix}
- q_1^T - \\
\vdots \\
- q_{n_q}^T -
\end{bmatrix} \\

\underset{d_k \times n_k}{K^T}
&= \begin{bmatrix}
| &  & | \\
k_1 & \cdots & k_{n_k} \\
| &  & | \\
\end{bmatrix} \\

\underset{n_k \times d_v}{V}
&= \begin{bmatrix}
- v_1^T - \\
\vdots \\
- v_{n_k}^T -
\end{bmatrix} \\
\end{align}
$$

* $Q \in \mathbb{R}^{n_q \times d_k}$, from the target sequence
* $K \in \mathbb{R}^{n_k \times d_k}$, from the source sequence
* $V \in \mathbb{R}^{n_k \times d_v}$, from the source sequence, $d_v$ is the
  dimension of each value vector in $V$, it does not need to be the same as
  $d_k$, but in practice, esp. in multi-head setup, $d_v = d_k =
  d_{\text{model}}/h$, where $d_{\text{model}}$ is the embedding size, and $h$
  is the number of attention heads.


$$
\begin{align*}
\underset{n_q \times n_k}{Q K^T}
&= \begin{bmatrix}
- q_1^T - \\
\vdots \\
- q_{n_q}^T -
\end{bmatrix} \begin{bmatrix}
| &  & | \\
k_1 & \cdots & k_{n_k} \\
| &  & | \\
\end{bmatrix} \\
&= \begin{bmatrix}
q_1^T k_1 & \cdots & q_1^T k_{n_k}  \\
\vdots & \ddots & \vdots \\
q_{n_q}^T k_1 & \cdots & q_{n_q}^T k_{n_k}  \\
\end{bmatrix}
\end{align*}
$$

Then, **attention** is defined as

$$
\underset{n_q \times d_v}{\text{Attn}(Q, K, V)} = \text{softmax}\left(\frac{Q
K^T}{\sqrt{d_k}} \right)V
$$

Note,

* $QK^T \in \mathbb{R}^{n_q \times n_k}$, so the output is $\text{Attn}(Q, K, V)
  \in \mathbb{R}^{n_q \times d_v}$
* the softmax should be applied along the axis of $n_k$ so that we obtain a
  weight for each of the $n_k$ elements in the source sequence.